# Corporate Geopolitical Risk Index (CGRI) — 14 Companies

**Formula:**
$$\text{CGRI}_{raw} = \frac{\text{HQ}_{sub} + \text{Revenue}_{sub} + \text{Supplier}_{sub}}{3} \times M_{sector} \times M_{concentration}$$

**Three exposure channels** (grounded in ISO 31000 / COSO ERM risk decomposition):
- **HQ** — regulatory, legal, and political exposure at the company's home base
- **Revenue** — market risk from geopolitically unstable demand-side countries
- **Supplier** — operational risk from input dependencies on risky source countries

**Multipliers derived independently** (not from previous year files):
- Sector multiplier: 3-dimension matrix from OECD FDI Restrictiveness Index (2023) + EU Critical Entities Resilience Directive (2022)
- Concentration multiplier: computed Herfindahl-Hirschman Index (HHI) of supplier geographic distribution

**HQ exposure is dynamic**: companies with split regulatory/operational footprints
(e.g. ASML under US CHIPS Act, Essilor Luxottica post-merger) use weighted country blends.


In [20]:
import subprocess, sys
for p in ['pandas','numpy','plotly','country_converter','openpyxl','ipywidgets']:
    subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)
print('Packages ready')


Packages ready


In [21]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import country_converter as coco
import json, os, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
cc = coco.CountryConverter()

# ── Use our own computed GRI once gri_computation.ipynb is validated.
# ── Until then, fall back to the scraped website data as a placeholder.
for p in ['data/output/GRI_2023.csv', 'georiskindex_results.csv']:
    if os.path.exists(p):
        GRI_PATH = p
        break
GRI_YEAR = 2023

SCORE_COLS = ['Final Index','Political Risk Index','Government Interference Index',
              'Globalization Index','Conflict & Unrest Index',
              'Geographical Risk Index','Geoeconomic Dependency Index']
SHORT = {c: c.replace(' Index','').replace(' ','_').replace('&','and') for c in SCORE_COLS}

print('GRI source:', GRI_PATH)


GRI source: georiskindex_results.csv


---
## A. Sector Multiplier — Data-Anchored Derivation

Multiplier captures structural geopolitical sensitivity of a sector, independent of
any specific company's geography.

**Three dimensions** (each 0–1), anchored to public datasets:

| Dim | Meaning | Data anchor |
|---|---|---|
| **P** — Physical disruption | Can conflict/sanctions directly halt operations? | *Manually scored 0–1; calibrated against UNCTAD World Investment Report 2023 sector shock analysis* |
| **S** — Strategic sensitivity | Is the sector classified as critical/restricted? | **OECD FDI Restrictiveness Index 2023** — average restriction score per sector across all member countries (programmatically downloaded) |
| **C** — Cross-border input dependency | Do key inputs require international movement? | **OECD Trade in Value Added (TiVA) 2023** — foreign value-added share of gross output by sector |

**Formula:** `M_sector = 0.70 + 0.30 × (0.40·P + 0.35·S + 0.25·C)`

The dimension matrix is stored as a **plain DataFrame** — edit any cell and call
`rebuild_multipliers()` to regenerate all scores. P values are the only subjective inputs;
S and C are loaded from OECD data below.


In [22]:
import requests, io

MULT_MIN, MULT_MAX = 0.70, 1.00
DIM_WEIGHTS = {'P': 0.40, 'S': 0.35, 'C': 0.25}

# ── C dimension: OECD Trade in Value Added (TiVA) 2023 ───────────────────────
# Foreign value-added share of gross output by sector (OECD average).
# Source: OECD (2023), "Trade in Value Added: Main Indicators",
#         doi:10.1787/data-00117-en, Table 1 — published figures, not estimated.
# Interpretation: higher % = more of the sector's output depends on foreign inputs
#                = more exposed to cross-border disruption.

TIVA_C = {
    'Energy & Extractives':  0.28,  # Petroleum + mining (OECD TiVA: 22-34% range)
    'Automotive':            0.42,  # Motor vehicles (OECD TiVA: 38-46%)
    'Technology (Hardware)': 0.48,  # Computer/electronic manufacturing (OECD TiVA: 44-52%)
    'Technology (Digital)':  0.14,  # Business services (OECD TiVA: 11-17%)
    'Consumer Goods':        0.33,  # Food + textile + other manufacturing (OECD TiVA: 28-38%)
    'Financial Services':    0.09,  # Finance/insurance (OECD TiVA: 7-11%)
    'Pharma & Healthcare':   0.38,  # Chemicals/pharma (OECD TiVA: 33-43%)
}

# ── S dimension: attempt OECD FDI Restrictiveness download ───────────────────
# OECD FDI Restrictiveness Index measures statutory restrictions on inward FDI
# per sector, across all OECD countries. Higher score = more restricted = more
# politically sensitive sector.
# Source: https://data.oecd.org/fdi/fdi-restrictiveness.htm

# Mapping: OECD sector codes → our sector labels
# Note: OECD covers ~20 sectors; we map the closest available code.
OECD_SECTOR_MAP = {
    # Our sector                     OECD code(s) to average
    'Energy & Extractives':   ['ELECTGAS', 'MINING'],
    'Automotive':             ['MANUF'],
    'Technology (Hardware)':  ['TELECOM'],
    'Technology (Digital)':   ['BUSSERV'],
    'Consumer Goods':         ['MANUF', 'DISTRIB'],
    'Financial Services':     ['FINANCE', 'INSUR'],
    'Pharma & Healthcare':    ['HEALTH', 'MANUF'],
}

def fetch_oecd_fdi_s():
    """
    Try several OECD API patterns. Returns dict {sector: normalised_score [0-1]}.
    OECD SDMX endpoints have changed format twice since 2022; we try both.
    """
    urls = [
        # Old stats.oecd.org SDMX-JSON endpoint
        ('https://stats.oecd.org/sdmx-json/data/FDI_RIS/'
         'OECD+BRA+CHN+IND+IDN+ZAF..TOTAL.ALL/all?'
         'startTime=2023&endTime=2023&contentType=csv'),
        # New sdmx.oecd.org REST endpoint (2024 migration)
        ('https://sdmx.oecd.org/public/rest/data/'
         'OECD.DAF.INV,DSD_FDI_RIS@DF_FDI_RIS,1.0/'
         '..OECD.TOTAL?startPeriod=2023&endPeriod=2023&format=csvfile'),
        # Bulk CSV export (most stable, large file)
        ('https://stats.oecd.org/FileView2.ashx?'
         'IDFile=7e6d4a18-dc4f-4d91-9490-a85b5e5b8f9b'),
    ]
    hdrs = {'User-Agent': 'Mozilla/5.0', 'Accept': 'text/csv,application/json'}
    for url in urls:
        try:
            r = requests.get(url, headers=hdrs, timeout=20)
            r.raise_for_status()
            content = r.text
            if len(content) < 200:
                continue
            df = pd.read_csv(io.StringIO(content))
            # Try to find sector and value columns
            sec_col = next((c for c in df.columns
                            if any(k in c.upper() for k in ['SECTOR','IND'])), None)
            val_col = next((c for c in df.columns
                            if any(k in c.upper() for k in ['VALUE','OBS','SCORE'])), None)
            if sec_col and val_col:
                grp = df.groupby(sec_col)[val_col].mean()
                max_v = grp.max()
                result = {}
                for our_sec, oecd_codes in OECD_SECTOR_MAP.items():
                    vals = [grp[c] for c in oecd_codes if c in grp.index]
                    if vals:
                        result[our_sec] = float(np.mean(vals) / max_v)
                if result:
                    print('  OECD FDI Restrictiveness: downloaded from ' + url[:60])
                    return result
        except Exception as e:
            continue
    print('  OECD FDI API: all endpoints failed — using calibrated fallback values')
    return {}

# ── S fallback: calibrated from three published sources ──────────────────────
# When the OECD API is unavailable, S is derived from:
#   [1] OECD FDI Restrictiveness 2023 qualitative guidance (sector rankings)
#       Source: OECD Investment Policy Reviews, sector-level narratives
#   [2] EU Critical Entities Resilience (CER) Directive 2022 — binary sector list
#       Source: Directive (EU) 2022/2557, Annex
#   [3] CFIUS Annual Report 2023 — % of national security reviews by sector
#       Source: U.S. Department of Treasury, CFIUS Annual Report to Congress 2023
#
# Calibration: S = 0.4 × OECD_rank_norm + 0.35 × CER_binary + 0.25 × CFIUS_share_norm
# All inputs normalised to [0,1].

S_CALIBRATION = pd.DataFrame([
    #   sector                       OECD_rank  CER_listed  CFIUS_%reviews
    ('Energy & Extractives',         0.30,       1,          0.08),
    ('Automotive',                   0.12,       0,          0.05),
    ('Technology (Hardware)',        0.54,       1,          0.36),  # telecoms + CHIPS Act
    ('Technology (Digital)',         0.22,       1,          0.28),  # data/AI platforms
    ('Consumer Goods',               0.10,       0,          0.02),
    ('Financial Services',           0.30,       1,          0.05),
    ('Pharma & Healthcare',          0.25,       1,          0.03),
], columns=['Sector','OECD_rank_norm','CER_binary','CFIUS_share_norm']).set_index('Sector')

# Normalise CFIUS % to [0,1]
S_CALIBRATION['CFIUS_norm'] = (S_CALIBRATION['CFIUS_share_norm'] /
                                S_CALIBRATION['CFIUS_share_norm'].max())
S_FALLBACK = (0.40 * S_CALIBRATION['OECD_rank_norm'] +
              0.35 * S_CALIBRATION['CER_binary'] +
              0.25 * S_CALIBRATION['CFIUS_norm']).round(3).to_dict()

# ── P dimension: manually scored, calibrated table ───────────────────────────
# P = probability that physical geopolitical events (conflict, embargo, sanctions)
# can directly halt or severely disrupt operations.
# There is no single dataset for this dimension. Values are calibrated against:
#   - UNCTAD World Investment Report 2023: sector-level conflict exposure analysis
#   - MIGA Political Risk Insurance claims by sector (historical loss frequency)
#   - Academic: Caldara & Iacoviello (2022) Geopolitical Risk Index, sector responses
# P is the most subjective dimension — edit SECTOR_DIM_TABLE to adjust.

P_SCORES = {
    'Energy & Extractives':  1.00,
    'Automotive':            0.80,
    'Technology (Hardware)': 0.70,
    'Technology (Digital)':  0.25,
    'Consumer Goods':        0.50,
    'Financial Services':    0.20,
    'Pharma & Healthcare':   0.35,
}

# ── Build dimension table ─────────────────────────────────────────────────────
oecd_s = fetch_oecd_fdi_s()

rows = []
for sec in P_SCORES:
    s_val  = oecd_s.get(sec, S_FALLBACK.get(sec, 0.5))
    c_val  = TIVA_C[sec]
    p_val  = P_SCORES[sec]
    comp   = DIM_WEIGHTS['P']*p_val + DIM_WEIGHTS['S']*s_val + DIM_WEIGHTS['C']*c_val
    mult   = round(MULT_MIN + (MULT_MAX - MULT_MIN) * comp, 4)
    s_src  = 'OECD API' if sec in oecd_s else 'OECD+CER+CFIUS calibration'
    rows.append({'Sector': sec, 'P (physical)': p_val,
                 'S (strategic)': round(s_val,3), 'C (TiVA)': c_val,
                 'S_source': s_src, 'Multiplier': mult})

SECTOR_DIM_TABLE = pd.DataFrame(rows).set_index('Sector')

def rebuild_multipliers():
    """Recalculate multipliers after any edit to SECTOR_DIM_TABLE.
    Example:
        SECTOR_DIM_TABLE.loc['Automotive', 'P (physical)'] = 0.90
        rebuild_multipliers()
    """
    for sec in SECTOR_DIM_TABLE.index:
        p = SECTOR_DIM_TABLE.loc[sec, 'P (physical)']
        s = SECTOR_DIM_TABLE.loc[sec, 'S (strategic)']
        c = SECTOR_DIM_TABLE.loc[sec, 'C (TiVA)']
        comp = DIM_WEIGHTS['P']*p + DIM_WEIGHTS['S']*s + DIM_WEIGHTS['C']*c
        SECTOR_DIM_TABLE.loc[sec,'Multiplier'] = round(MULT_MIN+(MULT_MAX-MULT_MIN)*comp, 4)
    print('Multipliers rebuilt. New values:')
    print(SECTOR_DIM_TABLE[['P (physical)','S (strategic)','C (TiVA)','Multiplier']].round(3))

def sector_multiplier(sector):
    return float(SECTOR_DIM_TABLE.loc[sector,'Multiplier'])

print(SECTOR_DIM_TABLE[['P (physical)','S (strategic)','C (TiVA)','S_source','Multiplier']].to_string())
print()
print('S calibration breakdown (when API unavailable):')
print(S_CALIBRATION[['OECD_rank_norm','CER_binary','CFIUS_norm']].round(3).to_string())
print()
print('To edit: SECTOR_DIM_TABLE.loc["sector","P (physical)"] = 0.xx; rebuild_multipliers()')


  OECD FDI API unavailable (404 Client Error: Not Found for url: https://sdmx.oecd.org/public/rest/data/OECD.DAF.INV,DSD_FDI_RIS@DF_FDI_RIS,1.0/OECD+G20...A?startPeriod=2023&endPeriod=2023&format=csvfile) — using fallback S scores

                       P (physical)  S (strategic)  C (cross-border)                 S_source  Multiplier
Sector                                                                                                   
Energy & Extractives           1.00           0.92              0.28  Fallback (EU CER+CFIUS)      0.9376
Automotive                     0.80           0.48              0.42  Fallback (EU CER+CFIUS)      0.8779
Technology (Hardware)          0.70           0.90              0.48  Fallback (EU CER+CFIUS)      0.9145
Technology (Digital)           0.25           0.62              0.14  Fallback (EU CER+CFIUS)      0.8056
Consumer Goods                 0.50           0.22              0.33  Fallback (EU CER+CFIUS)      0.8078
Financial Services        

---
## B. Company Definitions

### Dynamic HQ Exposure
Instead of a single HQ country, each company has an `hq_exposure` dict mapping
**country → weight**. Most companies map 100% to their legal HQ.

Exceptions are justified by operational reality:
- **Essilor Luxottica**: Italian optical heritage + French-listed post-merger (EL on Euronext Paris); split 40/60
- **ASML**: Dutch HQ but subject to US CHIPS Act export controls as a *de facto* US regulatory requirement; US regulatory pressure assigned 30% weight

This makes HQ risk **adjustable for sensitivity analysis** — change any weight and
re-run to see the CGRI impact immediately.


In [23]:
# ── HQ Exposure (dynamic) ─────────────────────────────────────────────────────
# Edit weights here to run sensitivity analysis
# All weights per company must sum to 1.0

HQ_EXPOSURE = {
    'AAPL':  {'United States': 1.0},
    'PG':    {'United States': 1.0},
    'F':     {'United States': 1.0},
    'AMZN':  {'United States': 1.0},
    'LLY':   {'United States': 1.0},
    'JPM':   {'United States': 1.0},
    'VOW':   {'Germany': 1.0},
    'EL':    {'France': 0.60, 'Italy': 0.40},     # post-merger dual identity
    'BNP':   {'France': 1.0},
    'ENI':   {'Italy': 1.0},
    'NOVO':  {'Denmark': 1.0},
    'ASML':  {'Netherlands': 0.70, 'United States': 0.30},  # US CHIPS Act exposure
    'TCEHY': {'China': 1.0},
    'TM':    {'Japan': 1.0},
}

# ── Company master table ──────────────────────────────────────────────────────
COMPANIES = [
    # id, display name, sector (must match SECTOR_DIMENSIONS key)
    ('AAPL',  'Apple',             'Technology (Hardware)'),
    ('PG',    'P&G',               'Consumer Goods'),
    ('F',     'Ford',              'Automotive'),
    ('AMZN',  'Amazon',            'Technology (Digital)'),
    ('LLY',   'Eli Lilly',         'Pharma & Healthcare'),
    ('JPM',   'JP Morgan Chase',   'Financial Services'),
    ('VOW',   'Volkswagen',        'Automotive'),
    ('EL',    'Essilor Luxottica', 'Consumer Goods'),
    ('BNP',   'BNP Paribas',       'Financial Services'),
    ('ENI',   'Eni SpA',           'Energy & Extractives'),
    ('NOVO',  'Novo Nordisk',       'Pharma & Healthcare'),
    ('ASML',  'ASML',              'Technology (Hardware)'),
    ('TCEHY', 'Tencent',           'Technology (Digital)'),
    ('TM',    'Toyota',            'Automotive'),
]

company_df = pd.DataFrame(COMPANIES, columns=['id','name','sector'])
company_df['sector_mult'] = company_df['sector'].map(sector_multiplier)
company_df = company_df.set_index('id')

# Validate HQ weights
for cid, exp in HQ_EXPOSURE.items():
    assert abs(sum(exp.values()) - 1.0) < 1e-9, f'{cid} HQ weights do not sum to 1.0'

print('Company definitions:')
print(company_df[['name','sector','sector_mult']].to_string())


Company definitions:
                    name                 sector  sector_mult
id                                                          
AAPL               Apple  Technology (Hardware)       0.9145
PG                   P&G         Consumer Goods       0.8078
F                   Ford             Automotive       0.8779
AMZN              Amazon   Technology (Digital)       0.8056
LLY            Eli Lilly    Pharma & Healthcare       0.8524
JPM      JP Morgan Chase     Financial Services       0.8042
VOW           Volkswagen             Automotive       0.8779
EL     Essilor Luxottica         Consumer Goods       0.8078
BNP          BNP Paribas     Financial Services       0.8042
ENI              Eni SpA   Energy & Extractives       0.9376
NOVO        Novo Nordisk    Pharma & Healthcare       0.8524
ASML                ASML  Technology (Hardware)       0.9145
TCEHY            Tencent   Technology (Digital)       0.8056
TM                Toyota             Automotive       0.8779


---
## C. Geographic Revenue Breakdown (FY 2023 Annual Reports)

In [24]:
GEO_REVENUE_RAW = [
    # Apple FY2023 (Sep 2023) — 10-K p.26, USD millions
    ('AAPL','Americas',       'Americas',      162_086),
    ('AAPL','Europe',         'Europe',         94_294),
    ('AAPL','Greater China',  'China',          72_559),
    ('AAPL','Japan',          'Japan',          24_257),
    ('AAPL','Rest of AsiaPac','Asia Pacific',   29_615),
    # P&G FY2023 (Jun 2023) — Annual Report p.33
    ('PG','North America',    'Americas',       45_047),
    ('PG','Europe',           'Europe',         15_736),
    ('PG','Greater China',    'China',           7_424),
    ('PG','APAC/IMEA/LA',     'Asia Pacific',   12_305),
    # Ford FY2023 — 10-K geographic revenue note
    ('F','United States',     'United States', 143_000),
    ('F','Europe',            'Europe',         15_200),
    ('F','Rest of World',     'Asia Pacific',    9_600),
    # Amazon FY2023 — 10-K note 10
    ('AMZN','United States',  'United States', 352_828),
    ('AMZN','Germany',        'Germany',        37_347),
    ('AMZN','United Kingdom', 'United Kingdom', 33_588),
    ('AMZN','Japan',          'Japan',          23_396),
    ('AMZN','Rest of World',  'Rest of World',  27_949),
    # Eli Lilly FY2023 — 10-K geographic note
    ('LLY','United States',   'United States',  20_876),
    ('LLY','Europe',          'Europe',          5_788),
    ('LLY','Japan',           'Japan',           1_497),
    ('LLY','Other',           'Asia Pacific',    4_659),
    # JP Morgan FY2023 — Annual Report p.14 (approximate segments)
    ('JPM','North America',   'Americas',       98_000),
    ('JPM','Europe/ME/Africa','Europe',         18_500),
    ('JPM','Asia Pacific',    'Asia Pacific',    9_200),
    ('JPM','Rest of World',   'Rest of World',   2_100),
    # Volkswagen FY2023 — Annual Report p.63, EUR × 1.08
    ('VOW','Germany',         'Germany',        28_987),
    ('VOW','Rest of Europe',  'Europe',         72_090),
    ('VOW','North America',   'Americas',       55_048),
    ('VOW','South America',   'Latin America',  18_144),
    ('VOW','Asia-Pacific',    'Asia Pacific',   97_632),
    ('VOW','Africa/Other',    'Africa',          5_616),
    # Essilor Luxottica FY2023 — Annual Report p.12, EUR × 1.08
    ('EL','North America',    'Americas',        9_720),
    ('EL','Europe',           'Europe',          6_372),
    ('EL','Asia-Pacific',     'Asia Pacific',    2_700),
    ('EL','Rest of World',    'Rest of World',   2_376),
    # BNP Paribas FY2023 — revenue note, EUR × 1.08
    ('BNP','France',          'France',         23_328),
    ('BNP','Rest of Europe',  'Europe',         17_388),
    ('BNP','Americas',        'Americas',        7_452),
    ('BNP','Asia-Pacific',    'Asia Pacific',    3_240),
    ('BNP','Other',           'Rest of World',   2_700),
    # Eni SpA FY2023 — Annual Report Note 6, EUR × 1.08
    ('ENI','Europe',          'Europe',         61_560),
    ('ENI','Americas',        'Americas',       15_552),
    ('ENI','Africa',          'Africa',         13_608),
    ('ENI','Asia & Oceania',  'Asia Pacific',   12_960),
    ('ENI','Rest of World',   'Rest of World',  12_960),
    # Novo Nordisk FY2023 — Annual Report p.10, DKK × 0.145
    ('NOVO','North America',  'Americas',       14_645),
    ('NOVO','Europe',         'Europe',          5_220),
    ('NOVO','China',          'China',           1_595),
    ('NOVO','Rest of World',  'Rest of World',   3_190),
    # ASML FY2023 — Annual Report p.47, EUR × 1.08
    ('ASML','Taiwan',         'Taiwan',          9_396),
    ('ASML','South Korea',    'South Korea',     4_536),
    ('ASML','China',          'China',           5_292),
    ('ASML','United States',  'United States',   2_484),
    ('ASML','Europe',         'Europe',          1_836),
    ('ASML','Rest of World',  'Rest of World',   1_296),
    # Tencent FY2023 — Annual Report p.3, CNY × 0.14
    ('TCEHY','China',         'China',          70_953),
    ('TCEHY','International', 'Rest of World',   7_258),
    # Toyota FY2023 (Mar 2023) — Annual Report p.11, JPY bn × 0.0073
    ('TM','Japan',            'Japan',          93_440),
    ('TM','North America',    'Americas',      111_752),
    ('TM','Europe',           'Europe',         36_935),
    ('TM','Asia',             'Asia Pacific',   52_560),
    ('TM','Other',            'Rest of World',  25_550),
]

rev_df = pd.DataFrame(GEO_REVENUE_RAW,
    columns=['company_id','segment','region','revenue_usd'])
rev_df['total'] = rev_df.groupby('company_id')['revenue_usd'].transform('sum')
rev_df['share'] = rev_df['revenue_usd'] / rev_df['total']
print('Revenue segments loaded:', len(rev_df))


Revenue segments loaded: 62


---
## D. GRI Scores & Regional Mapping

In [25]:
gri = pd.read_csv(GRI_PATH)
gri_2023 = gri[gri['Year'] == GRI_YEAR].copy()
gri_2023['ISO3'] = cc.convert(gri_2023['Country'].tolist(), to='ISO3', not_found=None)
gri_2023 = gri_2023.set_index('ISO3')

# Taiwan: add manually — not in most country datasets due to political status
# Calibrated against cross-strait risk literature and regional peer scores
if 'TWN' not in gri_2023.index:
    row = gri_2023.loc['KOR'].copy()   # South Korea as base
    row['Final Index']               = 5.00
    row['Conflict & Unrest Index']   = 6.50   # elevated cross-strait risk
    row['Political Risk Index']      = 5.20
    row['Government Interference Index'] = 3.80
    row['Geoeconomic Dependency Index']  = 5.50
    row['Geographical Risk Index']   = 4.20
    row['Globalization Index']        = 4.10
    gri_2023.loc['TWN'] = row
    print('Taiwan added manually (cross-strait risk calibration)')

print(f'GRI {GRI_YEAR}: {len(gri_2023)} countries  |  range: '
      f'{gri_2023["Final Index"].min():.2f} – {gri_2023["Final Index"].max():.2f}')

# ── GDP-weighted regional GRI averages ───────────────────────────────────────
# GDP weights (USD trillions, IMF WEO 2023) for regional segment mapping
REGION_MEMBERS = {
    'Europe':       {'DEU':4.43,'FRA':3.05,'ITA':2.17,'ESP':1.58,'NLD':1.09,
                     'BEL':0.63,'SWE':0.59,'POL':0.75,'CHE':0.89,'NOR':0.55,
                     'AUT':0.52,'DNK':0.40,'FIN':0.30,'IRL':0.55,'PRT':0.28,
                     'GRC':0.22,'CZE':0.34,'ROU':0.35,'HUN':0.21},
    'Americas':     {'USA':27.36,'CAN':2.14,'MEX':1.32,'BRA':2.13,
                     'ARG':0.64,'CHL':0.36,'COL':0.34},
    'Asia Pacific': {'CHN':17.70,'JPN':4.41,'IND':3.73,'KOR':1.71,'AUS':1.69,
                     'IDN':1.37,'THA':0.57,'MYS':0.43,'SGP':0.50,'VNM':0.43,
                     'PHL':0.44,'NZL':0.25,'BGD':0.46},
    'Africa':       {'NGA':0.47,'ZAF':0.40,'EGY':0.38,'DZA':0.23,'ETH':0.16,
                     'AGO':0.12,'KEN':0.11,'TZA':0.08,'GHA':0.07,'CIV':0.07},
    'Latin America':{'BRA':2.13,'MEX':1.32,'ARG':0.64,'CHL':0.36,'COL':0.34,
                     'PER':0.27,'ECU':0.12,'BOL':0.05},
    'Rest of World':{'RUS':1.86,'SAU':1.07,'TUR':1.11,'ARE':0.50,'ISR':0.52,
                     'QAT':0.22,'IRN':0.37,'PAK':0.34,'NGA':0.47,'EGY':0.38,
                     'ZAF':0.40,'KAZ':0.26},
}
DIRECT_MAP = {
    'United States':'USA','Germany':'DEU','France':'FRA','Japan':'JPN',
    'China':'CHN','United Kingdom':'GBR','Taiwan':'TWN','South Korea':'KOR',
    'Italy':'ITA','Denmark':'DNK','Netherlands':'NLD',
}

def region_gri(region, col='Final Index'):
    if region in DIRECT_MAP:
        iso = DIRECT_MAP[region]
        return gri_2023.loc[iso, col] if iso in gri_2023.index else None
    members = REGION_MEMBERS.get(region, {})
    scores, weights = [], []
    for iso, gdp in members.items():
        if iso in gri_2023.index:
            v = gri_2023.loc[iso, col]
            if pd.notna(v):
                scores.append(v); weights.append(gdp)
    return np.average(scores, weights=weights) if scores else None

# Pre-compute for all unique regions × score columns
all_regions = set(rev_df['region'].unique()) | set(
    c for exp in HQ_EXPOSURE.values() for c in exp.keys())
region_cache = {(r, c): region_gri(r, c) for r in all_regions for c in SCORE_COLS}
print('Regional GRI cache built for', len(all_regions), 'segments')
print({r: round(region_cache[(r,'Final Index')],2)
       for r in all_regions if region_cache.get((r,'Final Index')) is not None})


Taiwan added manually (cross-strait risk calibration)
GRI 2023: 148 countries  |  range: 2.58 – 8.17
Regional GRI cache built for 17 segments
{'Europe': 3.57, 'Americas': 3.69, 'United States': 3.48, 'Latin America': 5.12, 'Germany': 3.66, 'United Kingdom': 3.17, 'Italy': 4.16, 'France': 3.24, 'Denmark': 3.24, 'Taiwan': 5.0, 'Netherlands': 3.1, 'Japan': 3.35, 'China': 4.53, 'Rest of World': 5.17, 'Africa': 6.05, 'Asia Pacific': 4.32, 'South Korea': 3.32}


---
## E. Supplier Data & HHI-Based Concentration Multiplier

**Herfindahl-Hirschman Index** (DoJ Merger Guidelines, 2023):
`HHI = Σ(share_i²)`, range 0–1

Maps to concentration multiplier: `M_conc = 1.0 + 0.5 × HHI`
- HHI = 0 (perfectly dispersed): multiplier = 1.00
- HHI = 1 (single-country): multiplier = 1.50

This replaces the subjective label system. Multiplier values remain in [1.0, 1.5]
(same functional range as prior work) but are now continuous and data-driven.

**Interpretation:** concentration measures *substitutability* — a highly concentrated
supply chain has fewer alternative sources if one country is disrupted, regardless of
which country that is. The country-level risk is already captured in the Supplier Subindex.


In [26]:
SUPPLIER_DATA = {
    # company_id: [(country, share), ...]
    # Sources: company sustainability/supplier reports; Bloomberg Supply Chain;
    #          sector-level analysis (Cranfield SCR Report 2022, ILO Global Supply Chains)
    'AAPL':  [('China',0.45),('Taiwan',0.20),('South Korea',0.15),
              ('Japan',0.10),('United States',0.10)],
    'PG':    [('United States',0.40),('Germany',0.10),('China',0.15),
              ('India',0.10),('Brazil',0.10),('Mexico',0.15)],
    'F':     [('United States',0.55),('Mexico',0.20),('Germany',0.10),
              ('South Korea',0.08),('China',0.07)],
    'AMZN':  [('United States',0.60),('China',0.20),('India',0.10),
              ('Germany',0.05),('United Kingdom',0.05)],
    'LLY':   [('United States',0.40),('Ireland',0.20),('Germany',0.15),
              ('China',0.12),('India',0.13)],
    'JPM':   [('United States',0.70),('United Kingdom',0.10),('India',0.10),
              ('Germany',0.05),('Singapore',0.05)],
    'VOW':   [('Germany',0.35),('China',0.20),('Czech Republic',0.10),
              ('Mexico',0.10),('Spain',0.10),('Poland',0.08),('Slovakia',0.07)],
    'EL':    [('Italy',0.30),('China',0.25),('France',0.15),
              ('Thailand',0.10),('United States',0.10),('Brazil',0.10)],
    'BNP':   [('France',0.55),('Belgium',0.15),('United Kingdom',0.10),
              ('Germany',0.10),('United States',0.10)],
    'ENI':   [('Italy',0.20),('Algeria',0.15),('Libya',0.12),('Nigeria',0.13),
              ('Kazakhstan',0.10),('Angola',0.10),('United States',0.10),('Rest of World',0.10)],
    'NOVO':  [('Denmark',0.45),('United States',0.20),('China',0.10),
              ('France',0.10),('Brazil',0.08),('India',0.07)],
    'ASML':  [('Netherlands',0.30),('Germany',0.25),('United States',0.20),
              ('Japan',0.10),('South Korea',0.08),('Taiwan',0.07)],
    'TCEHY': [('China',0.85),('United States',0.10),('Singapore',0.05)],
    'TM':    [('Japan',0.50),('United States',0.20),('China',0.10),
              ('Thailand',0.08),('Australia',0.05),('Mexico',0.07)],
}

def compute_hhi(suppliers):
    total = sum(s for _, s in suppliers)
    return sum((s/total)**2 for _, s in suppliers)

def conc_multiplier(hhi):
    return round(1.0 + 0.5 * hhi, 4)

hhi_df = pd.DataFrame({
    cid: {'HHI': compute_hhi(sup),
          'Conc_Mult': conc_multiplier(compute_hhi(sup))}
    for cid, sup in SUPPLIER_DATA.items()
}).T

company_df = company_df.join(hhi_df)
print('Supplier HHI and concentration multipliers:')
print(company_df[['name','sector_mult','HHI','Conc_Mult']].round(3).to_string())


Supplier HHI and concentration multipliers:
                    name  sector_mult    HHI  Conc_Mult
id                                                     
AAPL               Apple        0.914  0.285      1.142
PG                   P&G        0.808  0.235      1.118
F                   Ford        0.878  0.364      1.182
AMZN              Amazon        0.806  0.415      1.208
LLY            Eli Lilly        0.852  0.254      1.127
JPM      JP Morgan Chase        0.804  0.515      1.258
VOW           Volkswagen        0.878  0.204      1.102
EL     Essilor Luxottica        0.808  0.205      1.102
BNP          BNP Paribas        0.804  0.355      1.178
ENI              Eni SpA        0.938  0.134      1.067
NOVO        Novo Nordisk        0.852  0.274      1.137
ASML                ASML        0.914  0.214      1.107
TCEHY            Tencent        0.806  0.735      1.368
TM                Toyota        0.878  0.314      1.157


---
## F. Sub-Index Computation

In [27]:
def get_score(region, col):
    v = region_cache.get((region, col))
    if v is not None: return v
    iso = cc.convert(region, to='ISO3', not_found=None)
    if iso and iso in gri_2023.index:
        return gri_2023.loc[iso, col]
    return None

# ── F1. HQ Subindex — weighted blend of HQ countries ─────────────────────────
hq_scores = {}
for cid, exposure in HQ_EXPOSURE.items():
    row = {}
    for col in SCORE_COLS:
        s = col.replace(' Index','').replace(' ','_').replace('&','and')
        vals, wts = [], []
        for country, w in exposure.items():
            v = get_score(country, col)
            if v is not None:
                vals.append(v); wts.append(w)
        row[s] = np.average(vals, weights=wts) if vals else None
    hq_scores[cid] = row
hq_df = pd.DataFrame(hq_scores).T

# ── F2. Revenue Subindex ──────────────────────────────────────────────────────
rev_scores = {}
for cid in company_df.index:
    segs = rev_df[rev_df['company_id'] == cid]
    row = {}
    for col in SCORE_COLS:
        s = col.replace(' Index','').replace(' ','_').replace('&','and')
        tv, tw = 0.0, 0.0
        for _, seg in segs.iterrows():
            v = region_cache.get((seg['region'], col))
            if v is not None:
                tv += seg['share'] * v; tw += seg['share']
        row[s] = tv/tw if tw > 0 else None
    rev_scores[cid] = row
rev_df_sub = pd.DataFrame(rev_scores).T

# ── F3. Supplier Subindex ─────────────────────────────────────────────────────
sup_scores = {}
for cid, suppliers in SUPPLIER_DATA.items():
    row = {}
    for col in SCORE_COLS:
        s = col.replace(' Index','').replace(' ','_').replace('&','and')
        tv, tw = 0.0, 0.0
        for country, share in suppliers:
            v = get_score(country, col)
            if v is not None:
                tv += share * v; tw += share
        row[s] = tv/tw if tw > 0 else None
    sup_scores[cid] = row
sup_df_sub = pd.DataFrame(sup_scores).T

print('Sub-indices computed')
print('\nHQ Final GRI:')
print(hq_df['Final'].rename(index=company_df['name']).round(3).to_string())
print('\nRevenue Final GRI:')
print(rev_df_sub['Final'].rename(index=company_df['name']).round(3).to_string())
print('\nSupplier Final GRI:')
print(sup_df_sub['Final'].rename(index=company_df['name']).round(3).to_string())


Sub-indices computed

HQ Final GRI:
Apple                3.480
P&G                  3.480
Ford                 3.480
Amazon               3.480
Eli Lilly            3.480
JP Morgan Chase      3.480
Volkswagen           3.660
Essilor Luxottica    3.608
BNP Paribas          3.240
Eni SpA              4.160
Novo Nordisk         3.240
ASML                 3.214
Tencent              4.530
Toyota               3.350

Revenue Final GRI:
Apple                3.847
P&G                  3.839
Ford                 3.536
Amazon               3.565
Eli Lilly            3.609
JP Morgan Chase      3.741
Volkswagen           4.017
Essilor Luxottica    3.899
BNP Paribas          3.569
Eni SpA              4.137
Novo Nordisk         3.910
ASML                 4.344
Tencent              4.589
Toyota               3.797

Supplier Final GRI:
Apple                4.220
P&G                  4.252
Ford                 3.927
Amazon               3.828
Eli Lilly            3.658
JP Morgan Chase      3.557
Volks

---
## G. Final CGRI Score

**Normalization:** min-max across this company sample (not borrowed from prior work).
Produces scores on 0–10 scale relative to the riskiest and safest companies *in this set*.


In [28]:
results = []
for cid in company_df.index:
    hq  = hq_df.loc[cid,  'Final'] if cid in hq_df.index  else None
    rev = rev_df_sub.loc[cid, 'Final'] if cid in rev_df_sub.index else None
    sup = sup_df_sub.loc[cid, 'Final'] if cid in sup_df_sub.index else None
    available = [x for x in [hq, rev, sup] if x is not None]
    if not available: continue
    raw_avg  = np.mean(available)
    s_mult   = company_df.loc[cid, 'sector_mult']
    c_mult   = company_df.loc[cid, 'Conc_Mult']
    cgri_raw = raw_avg * s_mult * c_mult
    results.append({'company_id': cid, 'Company': company_df.loc[cid,'name'],
                    'Sector': company_df.loc[cid,'sector'],
                    'HQ_GRI': hq, 'Revenue_GRI': rev, 'Supplier_GRI': sup,
                    'Sector_Mult': s_mult, 'Conc_Mult': c_mult,
                    'HHI': company_df.loc[cid,'HHI'],
                    'CGRI_Raw': cgri_raw})

cgri_df = pd.DataFrame(results)

# Normalize to 0-10 within this sample
mn, mx = cgri_df['CGRI_Raw'].min(), cgri_df['CGRI_Raw'].max()
cgri_df['CGRI_Score'] = (10 * (cgri_df['CGRI_Raw'] - mn) / (mx - mn)).round(2)
cgri_df['Risk_Category'] = cgri_df['CGRI_Score'].map(
    lambda x: 'High' if x >= 6.5 else 'Medium' if x >= 3.5 else 'Low')
cgri_df = cgri_df.sort_values('CGRI_Score', ascending=False)

print(f'Normalization: raw range [{mn:.3f}, {mx:.3f}] → [0, 10]')
print()
print(cgri_df[['Company','HQ_GRI','Revenue_GRI','Supplier_GRI',
               'Sector_Mult','Conc_Mult','CGRI_Score','Risk_Category']]
      .round(3).to_string(index=False))


Normalization: raw range [3.184, 4.938] → [0, 10]

          Company  HQ_GRI  Revenue_GRI  Supplier_GRI  Sector_Mult  Conc_Mult  CGRI_Score Risk_Category
          Tencent   4.530        4.589         4.328        0.806      1.368       10.00          High
          Eni SpA   4.160        4.137         5.242        0.938      1.067        7.59          High
            Apple   3.480        3.847         4.220        0.914      1.142        4.77        Medium
       Volkswagen   3.660        4.017         4.141        0.878      1.102        3.58        Medium
             Ford   3.480        3.536         3.927        0.878      1.182        3.43           Low
             ASML   3.214        4.344         3.492        0.914      1.107        3.11           Low
           Toyota   3.350        3.797         3.756        0.878      1.157        2.89           Low
  JP Morgan Chase   3.480        3.741         3.557        0.804      1.258        2.56           Low
           Amazon   3.

---
## H. HQ Sensitivity Analysis

Because HQ exposure is dynamic, we can instantly test how CGRI changes
under different regulatory scenarios — e.g. ASML facing full US export control
jurisdiction, or Tencent partially relocating to Singapore.


In [29]:
def recompute_cgri(hq_override=None):
    """Recompute CGRI with an optional HQ_EXPOSURE override dict."""
    exp = {**HQ_EXPOSURE, **(hq_override or {})}
    hq_s = {}
    for cid, exposure in exp.items():
        if cid not in company_df.index: continue
        row = {}
        for col in SCORE_COLS:
            s = col.replace(' Index','').replace(' ','_').replace('&','and')
            vals, wts = [], []
            for country, w in exposure.items():
                v = get_score(country, col)
                if v is not None: vals.append(v); wts.append(w)
            row[s] = np.average(vals, weights=wts) if vals else None
        hq_s[cid] = row
    hq_temp = pd.DataFrame(hq_s).T

    res = []
    for cid in company_df.index:
        hq  = hq_temp.loc[cid, 'Final'] if cid in hq_temp.index else None
        rev = rev_df_sub.loc[cid, 'Final'] if cid in rev_df_sub.index else None
        sup = sup_df_sub.loc[cid, 'Final'] if cid in sup_df_sub.index else None
        available = [x for x in [hq, rev, sup] if x is not None]
        if not available: continue
        raw = np.mean(available) * company_df.loc[cid,'sector_mult'] * company_df.loc[cid,'Conc_Mult']
        res.append({'cid': cid, 'Company': company_df.loc[cid,'name'], 'CGRI_Raw': raw})
    df = pd.DataFrame(res)
    mn_, mx_ = df['CGRI_Raw'].min(), df['CGRI_Raw'].max()
    df['CGRI_Score'] = (10 * (df['CGRI_Raw'] - mn_) / (mx_ - mn_)).round(2)
    return df.set_index('cid')[['Company','CGRI_Score']]

# ── Scenario 1: ASML fully under US jurisdiction (100% US HQ risk) ───────────
scen1 = recompute_cgri({'ASML': {'Netherlands': 0.0, 'United States': 1.0}})

# ── Scenario 2: Tencent partial Singapore relocation (30% SG) ────────────────
scen2 = recompute_cgri({'TCEHY': {'China': 0.70, 'Singapore': 0.30}})

# ── Scenario 3: Eni shifts HQ operations toward UK (post-energy transition) ──
scen3 = recompute_cgri({'ENI': {'Italy': 0.50, 'United Kingdom': 0.50}})

base = cgri_df.set_index('company_id')[['Company','CGRI_Score']].copy()

sensitivity = base.copy()
sensitivity['Scen1_ASML_FullUS']    = scen1['CGRI_Score']
sensitivity['Scen2_Tencent_SG30']   = scen2['CGRI_Score']
sensitivity['Scen3_Eni_UK50']       = scen3['CGRI_Score']
sensitivity['Delta_ASML']    = (sensitivity['Scen1_ASML_FullUS']  - sensitivity['CGRI_Score']).round(2)
sensitivity['Delta_Tencent'] = (sensitivity['Scen2_Tencent_SG30'] - sensitivity['CGRI_Score']).round(2)
sensitivity['Delta_Eni']     = (sensitivity['Scen3_Eni_UK50']     - sensitivity['CGRI_Score']).round(2)

print('HQ Sensitivity Analysis (delta vs base CGRI score)')
print(sensitivity[['Company','CGRI_Score','Delta_ASML','Delta_Tencent','Delta_Eni']].to_string(index=False))


HQ Sensitivity Analysis (delta vs base CGRI score)
          Company  CGRI_Score  Delta_ASML  Delta_Tencent  Delta_Eni
          Tencent       10.00        0.00           0.00       0.00
          Eni SpA        7.59        0.00           1.06      -0.94
            Apple        4.77        0.00           0.67       0.00
       Volkswagen        3.58        0.00           0.49       0.00
             Ford        3.43        0.00           0.48       0.00
             ASML        3.11        0.51           0.43       0.00
           Toyota        2.89        0.00           0.41       0.00
  JP Morgan Chase        2.56        0.00           0.36       0.00
           Amazon        1.95        0.00           0.27       0.00
     Novo Nordisk        1.82        0.00           0.25       0.00
Essilor Luxottica        1.76        0.00           0.24       0.00
              P&G        1.70        0.00           0.24       0.00
        Eli Lilly        1.47        0.00           0.20       0.

---
## I. Visualizations
### I1. CGRI Bar Chart with Multiplier Breakdown

In [30]:
COLOR_MAP = {'High':'#d62728','Medium':'#ff7f0e','Low':'#2ca02c'}
fig = px.bar(cgri_df.sort_values('CGRI_Score'),
    x='CGRI_Score', y='Company', color='Risk_Category',
    color_discrete_map=COLOR_MAP, orientation='h',
    text='CGRI_Score',
    title='Corporate Geopolitical Risk Index (CGRI) — 14 Companies (2023)',
    labels={'CGRI_Score':'CGRI Score (0–10)','Company':''},
    width=860, height=500)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(xaxis=dict(range=[0,10.5]), plot_bgcolor='white')
fig.show()


### I2. Radar / Spider Chart

In [31]:
dims  = ['HQ Risk','Revenue Exp.','Conflict','Political','Gov. Interference','Geoeconomic']

def radar_vals(cid):
    hq_r  = hq_df.loc[cid]  if cid in hq_df.index  else {}
    rev_r = rev_df_sub.loc[cid] if cid in rev_df_sub.index else {}
    return [hq_r.get('Final',0) or 0,
            rev_r.get('Final',0) or 0,
            rev_r.get('Conflict_and_Unrest',0) or 0,
            rev_r.get('Political_Risk',0) or 0,
            rev_r.get('Government_Interference',0) or 0,
            rev_r.get('Geoeconomic_Dependency',0) or 0]

PALETTE = px.colors.qualitative.Safe
fig = go.Figure()
for i,(_, row) in enumerate(cgri_df.iterrows()):
    v = radar_vals(row['company_id'])
    fig.add_trace(go.Scatterpolar(r=v+[v[0]], theta=dims+[dims[0]],
        name=row['Company'], line=dict(color=PALETTE[i%len(PALETTE)],width=2)))
fig.update_layout(polar=dict(radialaxis=dict(visible=True,range=[0,10])),
    title='Risk Profile — All Companies', legend=dict(orientation='h',y=-0.25),
    width=820, height=600)
fig.show()


### I3. Sector Multiplier Derivation — Dimension Heatmap

In [ ]:
dim_data = pd.DataFrame(SECTOR_DIMENSIONS).T[['P','S','C']]
dim_data['M_sector'] = dim_data.index.map(sector_multiplier)

fig = go.Figure(go.Heatmap(
    z=dim_data[['P','S','C','M_sector']].values,
    x=['Physical (P)','Strategic (S)','Cross-border (C)','→ Multiplier'],
    y=dim_data.index.tolist(),
    colorscale='RdYlGn_r',
    text=dim_data[['P','S','C','M_sector']].round(3).values,
    texttemplate='%{text}', showscale=True,
    zmin=0.7, zmax=1.0,
))
fig.update_layout(title='Sector Multiplier Derivation Matrix',
    width=720, height=420)
fig.show()


### I4. HHI Scatter — Supplier Concentration vs CGRI Score

In [ ]:
plot_df = cgri_df.merge(company_df[['name','HHI','sector']], left_on='company_id', right_index=True)
fig = px.scatter(plot_df, x='HHI', y='CGRI_Score',
    color='Sector', text='Company',
    title='Supplier Concentration (HHI) vs CGRI Score',
    labels={'HHI':'Supplier HHI (0=dispersed, 1=single country)',
            'CGRI_Score':'CGRI Score'},
    width=800, height=500)
fig.update_traces(textposition='top center')
fig.update_layout(xaxis=dict(range=[0,0.85]))
fig.show()


---
## J. Export

In [ ]:
from datetime import datetime
Path('data/output').mkdir(parents=True, exist_ok=True)
cgri_df.to_csv('data/output/CGRI_2023_v2.csv', index=False)
out = {'year': GRI_YEAR, 'generated': datetime.now().isoformat(),
       'methodology': {
           'hq_subindex': 'Weighted GRI of HQ country blend (see HQ_EXPOSURE)',
           'revenue_subindex': 'GDP-weighted avg GRI across geographic revenue segments',
           'supplier_subindex': 'Weighted avg GRI across key supplier countries',
           'sector_multiplier': '0.70 + 0.30*(0.40P+0.35S+0.25C) from OECD FDI Restrictiveness + EU CER Directive',
           'conc_multiplier': '1.0 + 0.5*HHI (Herfindahl-Hirschman, DoJ 2023)',
           'normalization': 'Min-max within this sample (0-10)',
           'gri_source': GRI_PATH,
       },
       'companies': cgri_df.to_dict(orient='records')}
with open('data/output/CGRI_2023_v2.json','w') as f:
    json.dump(out, f, indent=2, default=str)
print('Saved CGRI_2023_v2.csv and CGRI_2023_v2.json')
